# Master HHP Aortic Reproducibility Notebook

This notebook is the entry point for the public code archive. It preserves the QC supplementary notebook and regression notebook as separate artifacts, and it uses frozen controlled-access outputs or synthetic smoke-test files to recapitulate major analysis layers.

## 1. Notebook map

- QC supplementary notebook: `HHP_Aortic_QC_Supplementary_Methods_Reproducibility.ipynb`
- Regression figure notebook: `HHP_regression_figure_reproduction.ipynb`
- Authoritative scripts: `../analysis/*.py`

This master notebook does **not** duplicate those scripts.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
USE_SYNTHETIC_DEMO = True  # set False after placing approved files in data/controlled
DATA_DOI = "https://doi.org/10.5281/zenodo.20004811"
GITHUB_REPO = "https://github.com/sprakashUTH/ts-hhp-aortic-qc-regression"
DATA_DIR = ROOT / ('data/synthetic' if USE_SYNTHETIC_DEMO else 'data/controlled')
FIG_DIR = ROOT / 'figures'
FIG_DIR.mkdir(exist_ok=True)
print('Using data directory:', DATA_DIR)

## 2. Required files for complete manuscript recapitulation

The public archive contains synthetic smoke-test data only. Full numerical recapitulation requires approved de-identified files in `data/controlled/` as listed in `docs/DATA_REPOSITORY_SPECIFICATION.md`.

In [ ]:
required = [
    'aorta_data_long_with_outliers.csv',
    'qc_summary_by_patient_segment.csv',
    'patient_slopes_ols_vs_mixed.csv',
    'patient_progression_categories.csv',
    'patient_best_model_slopes_pooled_by_patient_segment.csv',
    'high_risk_combined_delta_AHI.csv',
    'ranked_participants.csv',
    'regression_analysis_dataset.csv',
    'mixed_model_long_dataset.csv',
    'early_growth_dataset.csv',
]
for f in required:
    p = DATA_DIR / (('synthetic_' + f) if USE_SYNTHETIC_DEMO and not (DATA_DIR/f).exists() else f)
    print(f, '->', 'FOUND' if p.exists() else 'MISSING')

## 3. Smoke-test: load available datasets

In [ ]:
def read_first_available(*names):
    for name in names:
        p = DATA_DIR / name
        if p.exists():
            return pd.read_csv(p), p
    raise FileNotFoundError(names)

reg, reg_path = read_first_available('regression_analysis_dataset.csv', 'synthetic_regression_analysis_dataset.csv')
print(reg_path)
reg.head()

## 4. Figures S1-S4 / S7-S12 recapitulation template

These panels require `regression_analysis_dataset.csv`. The synthetic data only demonstrate execution.

In [ ]:
# Select likely column names robustly
cols = reg.columns
segment_col = 'segment' if 'segment' in cols else None
x_col = 'baseline_diameter_cm' if 'baseline_diameter_cm' in cols else None
y_candidates = [c for c in ['best_slope_floored_mm_per_year','slope_floored_mm_per_year','best_model_slope_mm_per_year'] if c in cols]
y_col = y_candidates[0] if y_candidates else None

if segment_col and x_col and y_col:
    for seg, d in reg.dropna(subset=[x_col, y_col]).groupby(segment_col):
        plt.figure(figsize=(5,4))
        plt.scatter(d[x_col], d[y_col])
        if len(d) >= 2:
            z = np.polyfit(d[x_col], d[y_col], 1)
            xs = np.linspace(d[x_col].min(), d[x_col].max(), 50)
            plt.plot(xs, z[0]*xs + z[1])
        plt.title(f'Baseline diameter vs growth: {seg}')
        plt.xlabel('Baseline diameter (cm)')
        plt.ylabel('Growth rate (mm/year)')
        plt.tight_layout()
        plt.show()
else:
    print('Required columns not present in this smoke-test dataset:', { 'segment': segment_col, 'x': x_col, 'y': y_col })

## 5. QC notebook handoff

For detailed QC figures and supplementary-methods narrative, open `HHP_Aortic_QC_Supplementary_Methods_Reproducibility.ipynb`. That notebook is included unchanged from the QC package.

## 6. Regression notebook handoff

For regression figure panels and model-output visualizations, open `HHP_regression_figure_reproduction.ipynb` after placing controlled data files in `data/controlled/`.